# L13b: Interpretable Allocation and Model-Predictive Rebalancing
In this lecture, we turn the utility allocation of L8b into a book that trades every day. A live book has to decide which stocks to hold, how to weight them, and when to trade, while prices move and its estimates change. How do we build such a book, and how do we judge it when the real market gives us only one path?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Describe how the method chooses and weights stocks:__ Compute the γ score of a stock from its online single index model and the market state, describe the pickers, and pose the maximum-utility allocation with its limits on weight, volatility, turnover, and market loading.
> * __Explain when a model-predictive book re-plans:__ Describe the forecast band of a geometric Brownian motion forecast, the horizon and drawdown triggers, and the trading costs, and explain what the market hedge changes.
> * __Judge a trading method on real and synthetic markets:__ Explain why one historical path cannot rank a method's choices, and how typical and sell-off sets of synthetic markets test them.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
> * The score $\gamma_{i}(t)=\tanh\left(\tilde{g}_{i,t}/\max(|\beta_{i}|,0.1)^{\xi_{t}}\right)$ has the sign of the stock's expected growth rate $\tilde{g}_{i,t}$; the market state $\xi_{t}$ only changes its size, favoring stocks that move less with the market when it falls.
> * A re-plan maximizes a utility of the share counts, weighted by the scores, under a budget and limits on weight, volatility, and turnover. The book then holds its shares until the 21-day horizon passes, its value leaves the band of a forecast, or it falls 8% below its peak.
> * A backtest is one path. Synthetic markets fitted to the same stocks give many paths, and sell-off sets test the method on declines the real window never held.
>
> __For understanding__ (you will not need to reproduce or memorize these):
> * The exponentially weighted update of the single index model, the forecast covariance, the cost model, the hedge pool, and how the generator's β drift is fitted.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13b) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples
__We will work through these examples in lecture:__

> [▶ Run the INFORMS portfolio method on real data](CHEME-5660-L13b-Example-MPC-RealData-Fall-2026.ipynb). In this example, we calibrate the method on 2014 to 2024 for the 373 S&P 500 stocks with a complete history and trade 2025 and 2026. We follow the market state and the scores, run one book day by day and read why it re-planned, and then change the weighting rule, the picker, the hedge, and the basket on the same path.

> [▶ Run the INFORMS portfolio method on synthetic markets](CHEME-5660-L13b-Example-MPC-SyntheticData-Fall-2026.ipynb). In this example, we draw synthetic markets from a generator fitted to the same 373 stocks, choose typical markets and two kinds of sell-off as the study does, run the method on each market, and compare the books with `SPY` and an equal-weight portfolio set by set.

___

## From a Utility Allocation to a Live Book
In L8b, we chose share counts once, by maximizing a utility of the holdings under a budget, with preference weights from the single index model and the market. A book that trades for a year and a half meets problems that a single allocation does not:
* __What to hold:__ the universe has hundreds of stocks, and the estimates behind each preference weight change every day.
* __How to weight:__ a utility maximized under the budget alone can concentrate the book in a few stocks or take on more volatility than the market.
* __When to trade:__ trading every day costs money, and trading on a fixed calendar ignores what the market does in between.

This lecture follows the method of the INFORMS 2026 study (J. D. Varner, "Interpretable Allocation and Model-Predictive Rebalancing of Equity Portfolios, Tested on Real and Synthetic Markets", [arXiv:2611.03022](https://arxiv.org/abs/2611.03022)), which answers each question in a way we can inspect: a score from the single index model, a utility under explicit limits, and a controller that compares the book with a forecast. Its code is the [`MPCPortfolio` module of the course package](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/mpc/).

The study tests the method on daily data for the 373 S&P 500 stocks with a complete history from 2014 to 2026 and a sector: it calibrates on 2014 to 2024 and trades from January 2, 2025 to September 4, 2026, 420 trading days. Every decision uses data dated on or before the day it is made.
___

## What to Hold: the γ Score and the Picker
Every stock gets a score each day from its single index model, $g_{i}=\alpha_{i}+\beta_{i}\,g_{M}+\varepsilon_{i}$, where $g_{i}$ and $g_{M}$ are the daily growth rates of stock $i$ and of `SPY` (units: 1/yr). The model is fitted on 2014 to 2024 and then updated every trading day by exponentially weighted least squares with a half-life of 63 trading days, the online estimate of L7b, so $\alpha_{i}(t)$ and $\beta_{i}(t)$ follow the stock as it changes.

### The market inputs
Two inputs come from the `SPY` close, as in L8b:
* __Recent market growth:__ $\tilde{g}_{M,t}$ is the exponential moving average of the daily growth rate of `SPY` with a span of 252 trading days (units: 1/yr).
* __Market state:__ with $\bar{S}^{(21)}_{t}$ and $\bar{S}^{(63)}_{t}$ the exponential moving averages of the `SPY` close with spans of 21 and 63 trading days,
$$
\xi_{t} = -G\left(\frac{\bar{S}^{(21)}_{t}}{\bar{S}^{(63)}_{t}}-1\right)
$$
where the gain $G$ is chosen on 2014 to 2024 so that $|\xi_{t}|\leq 1$ on 99% of those days. The state is positive when the 21-day average lies below the 63-day average, as in a falling market, and negative in a rising one.

### The score
The score of stock $i$ on day $t$ is:
$$
\boxed{\gamma_{i}(t)=\tanh\left(\frac{\tilde{g}_{i,t}}{\max\left(|\beta_{i}(t)|,\,0.1\right)^{\xi_{t}}}\right),\qquad \tilde{g}_{i,t}=\alpha_{i}(t)+\beta_{i}(t)\,\tilde{g}_{M,t}}
$$
where $\tilde{g}_{i,t}$ is the stock's expected growth rate from its single index model, and the horizon of one year makes the argument of the $\tanh$ dimensionless. We read it as in L8b:
* __Sign:__ the denominator is positive, so $\gamma_{i}(t)$ has the sign of $\tilde{g}_{i,t}$. A stock with a positive expected growth rate has a positive score.
* __Size:__ when $\xi_{t}>0$ (falling market), the denominator is above one for a stock with $|\beta_{i}|>1$ and below one for a stock with $|\beta_{i}|<1$, so the score favors stocks that move less with the market. When $\xi_{t}<0$ (rising market), it favors stocks that move more. At $\xi_{t}=0$ the score is $\tanh(\tilde{g}_{i,t})$.

This is L8b's preference weight in a form that works for every stock. L8b assumed $\beta_{i}>0$; here $\max(|\beta_{i}|,0.1)$ keeps the denominator positive and finite for a stock whose $\beta_{i}$ is near zero or negative. A stock with $\beta_{i}<0$ is charged, not credited, for market growth through $\tilde{g}_{i,t}$.

### The pickers
The scores rank the stocks, and a __picker__ chooses the names $\mathcal{N}$ that the book may hold. The study compares several pickers; the ones in our examples are:
* __γ per sector:__ the 3 stocks with the highest $\gamma_{i}(t)$ in each of the 11 sectors, 33 names in all.
* __Random per sector:__ 3 stocks drawn at random in each sector, a baseline with the same sector balance.
* __A fixed list:__ names chosen in advance, such as the 13 stocks of L8b.

The picker runs on the first day, every 21 trading days after its previous pick, and whenever the book's value leaves its forecast band (below).
___

## How to Weight: Maximum Utility Under Limits
At a re-plan after the close of day $t$, the book chooses a number of shares $n_{i}\geq 0$ of every name in $\mathcal{N}$. With $S_{i}$ the close (USD/share), $W$ the wealth (USD), and $w_{i}=n_{i}S_{i}/W$ the weight of name $i$, it solves:
$$
\boxed{\begin{aligned}
\operatorname*{maximize}_{n\geq 0}\quad & U(n;\gamma)\\
\text{subject to}\quad & \sum_{i\in\mathcal{N}}n_{i}S_{i}\leq W && \text{(budget; the rest is cash)}\\
& w_{i}\leq\bar{w}_{i} && \text{(cap per name)}\\
& \sqrt{w^{\top}\Sigma_{g}\,w}\leq\sigma_{\text{SPY}} && \text{(volatility cap)}\\
& \bar{c}\,\bar{S}\sum_{i\in\mathcal{N}}\left|n_{i}-n_{i}^{\text{prev}}\right|\leq B && \text{(turnover budget)}
\end{aligned}}
$$
where:
* __Cap per name:__ $\bar{w}_{i}=\max(0.10,\,1.5/K)$ for a picker with $K$ names, so 10% of wealth for the 33-name pickers.
* __Volatility cap:__ $\Sigma_{g}$ (units: 1/yr²) is the covariance of the names' daily growth rates, the single index market term plus the residual covariance over the last 126 trading days, and $\sigma_{\text{SPY}}=2.741$ (1/yr) is the volatility of the daily growth rate of `SPY` on 2014 to 2024. The book may be no more volatile than the market.
* __Turnover budget:__ $n^{\text{prev}}$ are the shares held before the decision, $\bar{c}=5\times 10^{-4}$ is the half bid–ask spread, $\bar{S}$ is the mean price of the names, and $B=0.10\,W$.

### The utilities
The study compares seven weighting rules. Four are utilities of L8b's kind, built on the scores of the names with a positive score, $\mathcal{A}^{+}=\{i\in\mathcal{N}:\gamma_{i}>0\}$:
* __Cobb–Douglas:__ $U_{\text{CD}}=\prod_{i\in\mathcal{A}^{+}}n_{i}^{\gamma_{i}}$, which under the budget alone splits the invested wealth in proportion to $\gamma_{i}$, as in L8b.
* __CES on share counts:__ L8b's CES utility with elasticity $\eta=0.5$, whose share counts are proportional to $(\gamma_{i}/S_{i})^{\eta}$.
* __CES on weights:__ the same form in the weights $w_{i}$ instead of the share counts, so the allocation does not depend on the price of a share.
* __Stone–Geary:__ Cobb–Douglas above a baseline of 1% of wealth held in every name.

The other three are standard portfolios for comparison: minimum variance, the tangency portfolio, and a 130/30 long–short Cobb–Douglas book. Names with $\gamma_{i}\leq 0$ get no utility term, so the utility rules hold only names with a positive score.

Under the budget alone, each utility has the closed-form solution of L8b. With the limits, there is no closed form, but each problem is a convex optimization problem: the Cobb–Douglas and Stone–Geary utilities become exponential cones, the CES utilities power cones, the volatility cap a second-order cone, and the other limits are linear. The method solves it with the interior-point solver Clarabel, and falls back to SCS if Clarabel does not converge. If both fail, the book keeps its holdings.
___

## When to Trade: Model-Predictive Rebalancing
A re-plan sets target shares $n^{\star}$, and the plan is to hold them. A __model-predictive controller__ decides when the plan has failed. At each re-plan, it forecasts the book's wealth over the next $T=21$ trading days, and every day it compares the measured wealth with the forecast.

### The forecast band
The forecast is a multi-asset geometric Brownian motion written in growth rates. On each of $N_{p}=1{,}000$ simulated paths, the log price of every name advances each day by $\Delta t$ times a vector of growth rates drawn from a normal distribution with mean $\mu_{g}$, the names' expected growth rates $\tilde{g}_{i,t}$, and covariance $\Sigma_{g}$. The wealth of the plan on a path is the value of the shares $n^{\star}$ plus the leftover cash grown at the risk-free rate. Over the paths, the forecast wealth $\tau$ days after the re-plan has mean $\mu_{W}(\tau)$ and standard deviation $\sigma_{W}(\tau)$ (units: USD), and the band is:
$$
\mu_{W}(\tau)-z\,\sigma_{W}(\tau)\;\leq\; W(t)\;\leq\;\mu_{W}(\tau)+z\,\sigma_{W}(\tau),\qquad z=1.96
$$
which would hold the wealth on about 95% of days if the model were right.

### The triggers
After each close, with $\tau$ the trading days since the last re-plan and $W^{\text{pk}}$ the running peak of the wealth, the controller re-plans on the first of these that holds:
1. __Drawdown:__ $1-W(t)/W^{\text{pk}}>D_{\max}=0.08$. The peak is then reset to the current wealth, so the trigger fires once for each fresh loss of 8%.
2. __Horizon:__ $\tau\geq T$, the plan has run its 21 days.
3. __Band exit:__ the wealth has left the forecast band.

A scheduled pick that changes the names also starts a re-plan. Between re-plans the book does not trade. The band acts as a no-trade region placed on the value of the book relative to its forecast, and each re-plan starts a new forecast from the measured wealth, so the horizon recedes.

### Trades and costs
Orders fill at the next open, sells before buys, in fractional shares. A fill of $q$ shares of name $i$ at the open price $S^{o}_{i}$ costs:
$$
C(q)=\bar{c}\,S^{o}_{i}\,|q|+\kappa\,\frac{|q|}{A_{i}}\,S^{o}_{i}\,|q|
$$
half the bid–ask spread on the traded value, plus a market-impact cost that grows with the share of the stock's mean daily volume $A_{i}$ (shares/day) that the order takes, with $\kappa=0.001$.
___

## The Market Hedge
The method has an optional hedge. With the hedge on, every re-plan adds a limit on the book's market loading:
$$
\sum_{i\in\mathcal{N}}\beta_{i}\,w_{i}\;\leq\;\bar{\beta}=\max\left(0,\,1-\xi_{t}\right)
$$
The cap equals the loading of `SPY` when $\xi_{t}=0$, tightens as the market falls ($\xi_{t}>0$), reaches zero at $\xi_{t}\geq 1$, and loosens above one in a rising market. To meet a tight cap, the book can hold cash or names from a __hedge pool__: up to five stocks whose online $\beta_{i}$ and whose β on the market's down days over the last year are both at most zero. The pool joins every decision when the hedge is on.

The hedge trades growth in rising markets for protection in falling ones, so whether it pays depends on the market the book meets.
___

## How to Judge the Method
On real data, the method runs once: one sequence of prices from January 2025 to September 2026, with one broad sell-off in the spring of 2025. A choice that did well on that path may have been lucky, and one path cannot say which of two choices is better.

The study therefore also tests the method on synthetic markets from the multi-asset jump hidden Markov model generator of L13a, fitted to the same 373 stocks on 2014 to 2024. Two parts of the fit are tied to the method itself:
* __Drifting β:__ each stock's β drifts around its estimate, with a size chosen so that the method's own online estimate of β moves as much on synthetic stocks as on real ones.
* __Intercepts:__ each $\alpha_{i}$ is pulled toward the universe mean, so the synthetic intercepts spread and persist as the real ones do.

Each synthetic market covers the real calendar from 2014 to 2026. The method calibrates on its 2014 to 2024 and trades its 2025 and 2026, as on the real data. From 10,000 candidate markets, the study chooses three sets by the `SPY` path over the trading window:
* __Typical:__ 600 markets drawn at random.
* __Burst sell-offs:__ the 200 deepest drawdowns among the markets with a burst of extreme daily moves (a jump episode) in the window.
* __Bear sell-offs:__ the 200 deepest drawdowns among the markets without a burst, whose declines are slower and more sustained.

The results are then read as distributions, set by set, against two references on the same market: `SPY` bought and held, and an equal-weight portfolio of every stock.
___

## What the Study Found
The study reports, in its own words:
* __Stock choice versus weighting:__ "On real prices, the choice of stocks moved terminal wealth more than the choice among seven weighting rules: one 33-stock selection ended between 1.09 and 1.23 times initial wealth across the weighting rules, while different selections ended between 1.03 and 1.70, the highest being a 13-stock list the author picked in advance."
* __Early re-plans:__ The share of re-plans that came before the 21-day horizon ranged from a fifth to three fifths across combinations. Compared with re-planning every 21 trading days alone, these early re-plans "made the median worst drawdown smaller in 55 of the 70 combinations of the ten selection rules and seven weighting rules without the market hedge and in 61 with it, and raised trading cost in all of them, but raised median terminal wealth in 38 and 26 of the 70."
* __Sell-offs:__ "In both sets, the median terminal wealth of every combination ended above those of simulated SPY bought and held and of the equal-weight universe. Random selections did so as well." Because the equal-weight universe also ended above simulated `SPY`, part of the margin over `SPY` came from the simulated market itself.
* __Selection in typical futures:__ the 13 stocks with the highest scores ended above 13 random stocks in 62 to 71% of the typical futures, but the generator kept each stock's 2014 to 2024 trend, which the score estimates, so "part of this advantage came from the generator's design and may not carry over to real prices."
* __The hedge:__ it "reduced terminal wealth on real prices and added to it in the set without bursts, whose declines were the most sustained."

The study concludes that deciding which selection rule is better, and when such a hedge pays, "will need more historical windows and generators whose trends persist no more than real ones do."
___

## Summary
In this lecture, we built a book that chooses, weights, and trades stocks every day, and we saw how to test it on real and synthetic markets.

> __Key Takeaways:__
>
> * __A score from the single index model decides what to hold:__ We combined each stock's online single index model with the recent market growth and the market state into a bounded score. The score keeps the sign of the stock's expected growth rate and, when the market falls, favors stocks that move less with it; a picker keeps the highest-scoring stocks of each sector.
> * __A utility under limits sets the weights, and a forecast decides when to trade:__ We posed L8b's utility maximization with limits on weight, volatility, turnover, and, with the hedge, market loading, and solved it as a convex program. A controller then held the shares until the horizon passed, the wealth left the band of a forecast, or the book fell 8% from its peak.
> * __One path cannot judge a method, so we test it on many markets:__ We contrasted the real backtest with synthetic markets fitted to the same stocks and with sets of deep sell-offs. Reading results as distributions showed which conclusions hold across markets and which may come from the generator itself.

The method brings together the single index model, utility allocation, and synthetic markets from earlier lectures in one system that we can test on real and on synthetic data.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___